# UruTracker - Radar de Emenda Pix Parada

Identifica municipios com emenda Pix aprovada e obra parada, ranqueados por urgencia.
Dados: API Transferegov | Periodo: 2024+

## Parte 1 - Carga e Integracao dos Dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import date

DADOS = Path("../../../data_extraction")
HOJE = pd.Timestamp(date.today())

In [ ]:
plano_acao     = pd.read_csv(DADOS / "plano_acao_especial.csv", encoding="utf-8-sig")
plano_trabalho = pd.read_csv(DADOS / "plano_trabalho_especial.csv", encoding="utf-8-sig")
executor       = pd.read_csv(DADOS / "executor_especial.csv", encoding="utf-8-sig")
finalidade     = pd.read_csv(DADOS / "finalidade_especial.csv", encoding="utf-8-sig")

print(plano_acao.shape, plano_trabalho.shape, executor.shape, finalidade.shape)

In [ ]:
plano_trabalho["data_fim_execucao_plano_trabalho"] = pd.to_datetime(
    plano_trabalho["data_fim_execucao_plano_trabalho"], errors="coerce"
)

plano_acao["valor_total"] = (
    plano_acao["valor_custeio_plano_acao"].fillna(0)
    + plano_acao["valor_investimento_plano_acao"].fillna(0)
)

finalidade_agg = (
    finalidade.groupby("id_executor")["area_politica_publica_pt"]
    .apply(lambda x: " | ".join(sorted(x.dropna().unique())))
    .reset_index()
    .rename(columns={"area_politica_publica_pt": "setor"})
)

executor_primeiro = executor.drop_duplicates("id_plano_acao", keep="first")

In [ ]:
df = plano_acao.merge(plano_trabalho, on="id_plano_acao", how="left")
df = df.merge(executor_primeiro[["id_plano_acao", "id_executor", "objeto_executor"]], on="id_plano_acao", how="left")
df = df.merge(finalidade_agg, on="id_executor", how="left")

print(f"Total: {len(df)} emendas")
df.head(3)